<a href="https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Draft baseline rule

I will rank pages for human refresh review using only information available in the first 15 days of March 2026. The rule will give higher priority to pages that already have meaningful search visibility, have a relatively weak click-through rate, and are not ranking in the strongest position range. These pages may have enough audience impact to justify a human content review.

The output is a review queue, not an automatic content change. The action label is `human_refresh_review` and the single reason code is `visible_ctr_position_opportunity`.

This is a transparent baseline rule. It does not use `next_16d_impressions`, `is_next_15d_declining`, or any other future-window field. Those values are used only afterwards to evaluate whether the ranking was directionally useful.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("Token loaded safely.")

Token loaded safely.


In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connected. Working month: March 2026")

DuckDB connected. Working month: March 2026


In [ ]:
feature_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_impressions ELSE 0 END
            ) AS prior_15d_impressions,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_clicks ELSE 0 END
            ) AS prior_15d_clicks,

            AVG(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_avg_position END
            ) AS prior_15d_avg_position,

            COUNT(DISTINCT
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN report_date END
            ) AS prior_15d_active_days,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-16'
                                     AND DATE '2026-03-31'
                THEN gsc_impressions ELSE 0 END
            ) AS next_16d_impressions

        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        *,
        CASE
            WHEN prior_15d_impressions > 0
            THEN prior_15d_clicks / prior_15d_impressions
            ELSE 0
        END AS prior_15d_ctr,

        CASE
            WHEN prior_15d_impressions >= 100
             AND next_16d_impressions <= 0.80 * prior_15d_impressions
            THEN 1
            ELSE 0
        END AS is_next_15d_declining

    FROM page_windows
    WHERE prior_15d_impressions >= 100
""").df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(
    "Later-window decline proxy rate: "
    f"{feature_frame['is_next_15d_declining'].mean():.1%}"
)

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77,540
Later-window decline proxy rate: 28.6%


,client_hash_id,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_active_days,next_16d_impressions,prior_15d_ctr,is_next_15d_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,4.247255,15,711.0,0.004662,0
1,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,5.330069,15,793.0,0.001592,0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,4.468441,15,1490.0,0.007031,0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,2531.0,10.0,4.907404,15,3517.0,0.003951,0
4,client_73cda7b4e4f265ea,content_3dba50ae010f3f30,243.0,1.0,14.534282,15,114.0,0.004115,1
5,client_73cda7b4e4f265ea,content_098eedbd77ec1de1,176.0,0.0,36.527344,15,105.0,0.000000,1
6,client_73cda7b4e4f265ea,content_8935ed68eca88b01,2217.0,7.0,8.925289,15,2571.0,0.003157,0
7,client_73cda7b4e4f265ea,content_91ffe8aef1f8c426,129.0,0.0,7.432614,15,83.0,0.000000,1
8,client_73cda7b4e4f265ea,content_fd377b1af29dcbb3,195.0,0.0,30.304996,15,224.0,0.000000,0
9,client_73cda7b4e4f265ea,content_255f44daf21585b3,108.0,0.0,43.655700,15,120.0,0.000000,0


In [ ]:
baseline_df = feature_frame.copy()

baseline_df["visibility_bucket"] = pd.qcut(
    baseline_df["prior_15d_impressions"],
    q=4,
    duplicates="drop"
)

visibility_check = (
    baseline_df
    .groupby("visibility_bucket", observed=True)
    .agg(
        n=("is_next_15d_declining", "size"),
        median_prior_impressions=("prior_15d_impressions", "median"),
        later_decline_rate=("is_next_15d_declining", "mean")
    )
    .reset_index()
)

print("Signal 1 — prior visibility / volume")
display(visibility_check)

Signal 1 — prior visibility / volume


,visibility_bucket,n,median_prior_impressions,later_decline_rate
0,"(99.999, 238.0]",19460,154.0,0.292600
1,"(238.0, 576.0]",19349,368.0,0.290919
2,"(576.0, 1540.0]",19360,896.0,0.266684
3,"(1540.0, 161575.0]",19371,3114.0,0.292499


### Signal check 1 — prior visibility / volume

**Verdict: MIXED.** The observed later decline rates were similar across the four visibility buckets, ranging from 26.7% to 29.3%. Therefore, prior impressions alone did not strongly separate declining from non-declining pages in this slice. However, visibility remains useful for prioritisation because reviewing a highly visible page may have greater potential audience impact.

In [ ]:
baseline_df["position_bucket"] = pd.cut(
    baseline_df["prior_15d_avg_position"],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "positions_4_10",
        "positions_11_20",
        "positions_21_50",
        "below_50"
    ]
)

position_check = (
    baseline_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=("is_next_15d_declining", "size"),
        median_prior_position=("prior_15d_avg_position", "median"),
        later_decline_rate=("is_next_15d_declining", "mean")
    )
    .reset_index()
)

print("Signal 2 — prior average search position")
display(position_check)

Signal 2 — prior average search position


,position_bucket,n,median_prior_position,later_decline_rate
0,top_3,9558,2.229499,0.238858
1,positions_4_10,37383,5.623040,0.285879
2,positions_11_20,15172,13.631062,0.255339
3,positions_21_50,13849,28.767941,0.349412
4,below_50,1578,61.244912,0.297212


### Signal check 2 — prior average search position

**Verdict: CONFIRMED.** Pages in positions 21–50 had the highest observed later decline rate (34.9%), compared with 23.9% for pages in the top-three position bucket. The pattern was not perfectly monotonic across every bucket, but it directionally supports giving additional review priority to visible pages that rank outside the strongest position range.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os

score_df = baseline_df.copy()

# Transparent hand-written position weights.
# The signal check showed the 21–50 position range had the highest decline rate.
score_df["position_weight"] = np.select(
    [
        score_df["prior_15d_avg_position"] <= 3,
        score_df["prior_15d_avg_position"] <= 10,
        score_df["prior_15d_avg_position"] <= 20,
        score_df["prior_15d_avg_position"] <= 50
    ],
    [0.20, 0.50, 0.45, 1.00],
    default=0.35
)

# Lower CTR increases review priority, capped so that extreme values do not dominate.
score_df["low_ctr_weight"] = 1 - np.clip(
    score_df["prior_15d_ctr"] / 0.01,
    0,
    1
)

# Visibility gives more weight to pages where a review could affect more search users.
score_df["visibility_weight"] = np.log1p(
    score_df["prior_15d_impressions"]
)

# Hand-written baseline score: no future-window columns are used here.
score_df["baseline_action_score"] = (
    score_df["visibility_weight"]
    * (
        0.70 * score_df["position_weight"]
        + 0.30 * score_df["low_ctr_weight"]
    )
)

score_df["reason_code"] = "visible_ctr_position_opportunity"
score_df["action_label"] = "human_refresh_review"

ranked_eval = score_df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

def precision_at_k(frame, k):
    return frame.head(k)["is_next_15d_declining"].mean()

print(f"Baseline Precision@20: {precision_at_k(ranked_eval, 20):.3f}")
print(f"Baseline Precision@50: {precision_at_k(ranked_eval, 50):.3f}")
print(f"Overall later-window decline rate: {ranked_eval['is_next_15d_declining'].mean():.3f}")

# The exported queue contains only decision-time fields, not future outcomes.
queue_export = ranked_eval[
    [
        "client_hash_id",
        "content_hash_id",
        "prior_15d_impressions",
        "prior_15d_clicks",
        "prior_15d_avg_position",
        "prior_15d_ctr",
        "prior_15d_active_days",
        "baseline_action_score",
        "reason_code",
        "action_label"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

queue_export.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("\nCSV written: work/outputs/baseline_action_score.csv")
display(queue_export.head(20))

Baseline Precision@20: 0.750
Baseline Precision@50: 0.680
Overall later-window decline rate: 0.286

CSV written: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,baseline_action_score,reason_code,action_label
0,client_23a62021009f63c4,content_36e53e9c707674fc,109909.0,115.0,33.354423,0.001046,15,11.243065,visible_ctr_position_opportunity,human_refresh_review
1,client_23a62021009f63c4,content_559cdd76da9306de,62418.0,2.0,37.866104,0.000032,15,11.031011,visible_ctr_position_opportunity,human_refresh_review
2,client_23a62021009f63c4,content_bdf60c86117079be,60277.0,8.0,31.046299,0.000133,15,10.962898,visible_ctr_position_opportunity,human_refresh_review
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,84041.0,120.0,24.501281,0.001428,15,10.853349,visible_ctr_position_opportunity,human_refresh_review
4,client_23a62021009f63c4,content_9d1e94cbd32b0a41,50780.0,6.0,35.056300,0.000118,15,10.796870,visible_ctr_position_opportunity,human_refresh_review
5,client_23a62021009f63c4,content_573804af4f4fa09f,52948.0,21.0,29.926699,0.000397,15,10.747664,visible_ctr_position_opportunity,human_refresh_review
6,client_23a62021009f63c4,content_6486239516a186d7,49852.0,13.0,30.375806,0.000261,15,10.732212,visible_ctr_position_opportunity,human_refresh_review
7,client_23a62021009f63c4,content_99c63c59330193de,45442.0,5.0,37.406360,0.000110,15,10.688814,visible_ctr_position_opportunity,human_refresh_review
8,client_23a62021009f63c4,content_df47d1b976106de4,66342.0,83.0,25.221991,0.001251,15,10.685882,visible_ctr_position_opportunity,human_refresh_review
9,client_23a62021009f63c4,content_02474d1ff0eca1e5,49567.0,23.0,22.607521,0.000464,15,10.660604,visible_ctr_position_opportunity,human_refresh_review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20_review = queue_export.head(20).copy()
top20_review.insert(0, "rank", range(1, 21))

top20_review["confidence_note"] = (
    "Medium — high visibility, low CTR and a position outside the strongest range."
)

possible_failure_reasons = [
    "The page may already satisfy its search intent; low CTR may be caused by the SERP layout.",
    "The page may be seasonal, so the apparent opportunity may not persist.",
    "The page may target an intentionally narrow audience despite high impressions.",
    "A title or snippet test may be more appropriate than a full content refresh.",
    "The position may fluctuate naturally without a content-quality problem.",
    "The page may have strong conversion value even if its CTR is low.",
    "The page may be affected by competitor activity rather than stale content.",
    "The measured CTR may be driven by branded or unusual queries.",
    "The page may need technical investigation rather than editorial work.",
    "The page may already be scheduled for an update.",
    "The position may reflect a mismatch between query intent and the proposed action.",
    "The page may contain current information and not require a refresh.",
    "The observed pattern may be temporary search-demand variation.",
    "High impressions do not guarantee that a refresh would improve performance.",
    "The page may have a valid low CTR for its ranking range and topic.",
    "The page may be a duplicate or consolidation candidate instead of a refresh candidate.",
    "A SERP feature may be reducing clicks independently of page quality.",
    "The page may be visible for queries outside its intended topic.",
    "The action may be too costly relative to the page's real business value.",
    "The rule cannot inspect content quality, intent, or recent editorial changes."
]

top20_review["what_would_make_it_wrong"] = possible_failure_reasons

display(
    top20_review[
        [
            "rank",
            "content_hash_id",
            "action_label",
            "reason_code",
            "prior_15d_impressions",
            "prior_15d_avg_position",
            "prior_15d_ctr",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,action_label,reason_code,prior_15d_impressions,prior_15d_avg_position,prior_15d_ctr,confidence_note,what_would_make_it_wrong
0,1,content_36e53e9c707674fc,human_refresh_review,visible_ctr_position_opportunity,109909.0,33.354423,0.001046,"Medium — high visibility, low CTR and a positi...",The page may already satisfy its search intent...
1,2,content_559cdd76da9306de,human_refresh_review,visible_ctr_position_opportunity,62418.0,37.866104,0.000032,"Medium — high visibility, low CTR and a positi...","The page may be seasonal, so the apparent oppo..."
2,3,content_bdf60c86117079be,human_refresh_review,visible_ctr_position_opportunity,60277.0,31.046299,0.000133,"Medium — high visibility, low CTR and a positi...",The page may target an intentionally narrow au...
3,4,content_3df3f32f3fd58dea,human_refresh_review,visible_ctr_position_opportunity,84041.0,24.501281,0.001428,"Medium — high visibility, low CTR and a positi...",A title or snippet test may be more appropriat...
4,5,content_9d1e94cbd32b0a41,human_refresh_review,visible_ctr_position_opportunity,50780.0,35.056300,0.000118,"Medium — high visibility, low CTR and a positi...",The position may fluctuate naturally without a...
5,6,content_573804af4f4fa09f,human_refresh_review,visible_ctr_position_opportunity,52948.0,29.926699,0.000397,"Medium — high visibility, low CTR and a positi...",The page may have strong conversion value even...
6,7,content_6486239516a186d7,human_refresh_review,visible_ctr_position_opportunity,49852.0,30.375806,0.000261,"Medium — high visibility, low CTR and a positi...",The page may be affected by competitor activit...
7,8,content_99c63c59330193de,human_refresh_review,visible_ctr_position_opportunity,45442.0,37.406360,0.000110,"Medium — high visibility, low CTR and a positi...",The measured CTR may be driven by branded or u...
8,9,content_df47d1b976106de4,human_refresh_review,visible_ctr_position_opportunity,66342.0,25.221991,0.001251,"Medium — high visibility, low CTR and a positi...",The page may need technical investigation rath...
9,10,content_02474d1ff0eca1e5,human_refresh_review,visible_ctr_position_opportunity,49567.0,22.607521,0.000464,"Medium — high visibility, low CTR and a positi...",The page may already be scheduled for an update.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
used_score_columns = [
    "prior_15d_impressions",
    "prior_15d_avg_position",
    "prior_15d_ctr"
]

forbidden_future_columns = [
    "next_16d_impressions",
    "is_next_15d_declining"
]

print("Columns used to calculate the baseline score:")
print(used_score_columns)

print("\nLeakage check:")
leakage_found = [
    column for column in used_score_columns
    if column in forbidden_future_columns
]

if leakage_found:
    print("WARNING — leakage found:", leakage_found)
else:
    print("PASS — no future-window or label-derived columns were used in the score.")

print("\nTop-20 client concentration:")
top20_client_counts = (
    top20_review["client_hash_id"]
    .value_counts()
    .rename_axis("client_hash_id")
    .reset_index(name="pages_in_top20")
)

display(top20_client_counts)

print("\nPotentially weaker picks for human review:")
display(
    top20_review.tail(5)[
        [
            "rank",
            "content_hash_id",
            "prior_15d_impressions",
            "prior_15d_avg_position",
            "prior_15d_ctr",
            "what_would_make_it_wrong"
        ]
    ]
)

Columns used to calculate the baseline score:
['prior_15d_impressions', 'prior_15d_avg_position', 'prior_15d_ctr']

Leakage check:
PASS — no future-window or label-derived columns were used in the score.

Top-20 client concentration:


,client_hash_id,pages_in_top20
0,client_23a62021009f63c4,18
1,client_fef1a8f436438636,1
2,client_20259bd6705d81d4,1



Potentially weaker picks for human review:


,rank,content_hash_id,prior_15d_impressions,prior_15d_avg_position,prior_15d_ctr,what_would_make_it_wrong
15,16,content_c622f4ba7807c4be,41647.0,34.751241,0.000288,The page may be a duplicate or consolidation c...
16,17,content_ab91e088440ace78,38414.0,44.809035,0.000104,A SERP feature may be reducing clicks independ...
17,18,content_265c0bbf087c2194,38330.0,39.973135,0.000104,The page may be visible for queries outside it...
18,19,content_7d592b68cd59d19a,41433.0,29.064908,0.000386,The action may be too costly relative to the p...
19,20,content_b88025fbf2493889,45838.0,25.604808,0.000698,"The rule cannot inspect content quality, inten..."


### Weak picks and leakage check

The leakage check passed. The baseline score used only `prior_15d_impressions`, `prior_15d_avg_position`, and `prior_15d_ctr`, which were observable during the first 15 days of March. It did not use `next_16d_impressions` or `is_next_15d_declining`.

A limitation was observed in the ranked queue: 18 of the top 20 pages belonged to one pseudonymized client. This concentration may reflect that client's scale or search profile rather than a generally transferable refresh opportunity. Therefore, the queue should be reviewed by a human and, in a later version, may need a per-client cap or diversity rule.

The lower-ranked examples may be wrong because the rule cannot inspect search intent, content quality, seasonality, SERP features, conversion value, technical issues, or recent editorial changes. The action remains a human review recommendation, not an automatic refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.